# Sanskrit dataset analysis: Sansar corpus

This notebook inspects a small sample from the [Sansar Sanskrit Corpus](https://huggingface.co/datasets/MuseMesh/sansar-sanskrit-corpus). It streams at most 5,000 records so the full corpus is not downloaded.

The raw records remain unchanged in `raw_df`. Later, a separate processed copy preserves original text as `raw_text` and adds `normalized_text`. The normalizer applies Unicode NFC, standardizes horizontal whitespace and line endings, and preserves line breaks, punctuation, and source metadata, including licence and attribution fields.

Run cells in order with the project `.venv` selected as the kernel.

## 1. Imports and sample settings

The sample size is capped at 5,000. Streaming reads records incrementally rather than loading the complete corpus into memory.

In [ ]:
from collections import Counter
from itertools import islice
from pathlib import Path
import re
import sys
import unicodedata

import pandas as pd
from datasets import get_dataset_config_names, get_dataset_split_names, load_dataset

DATASET_ID = "MuseMesh/sansar-sanskrit-corpus"
DATASET_REVISION = "a268a87223c38575883c7ee6af25b1d2e65fd0d7"
CONFIG_NAME = "default"
SPLIT_NAME = "train"
SAMPLE_LIMIT = 5_000

# Support either the project root or notebooks/ as the working directory.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "requirements.txt").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the project or module directory.")

MODULE_ROOT = PROJECT_ROOT / "modules" / "module_01_normalization"
if str(MODULE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT / "src"))

from normalization import normalize_text

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_ID}")
print(f"Maximum sample size: {SAMPLE_LIMIT:,}")

## 2. Inspect configurations and splits

Check the repository metadata before reading records. The values below come from Hugging Face when the notebook runs.

In [ ]:
config_names = get_dataset_config_names(DATASET_ID, revision=DATASET_REVISION)
print("Available configurations:", config_names)

split_names_by_config = {}
for config_name in config_names:
    split_names = get_dataset_split_names(DATASET_ID, config_name=config_name, revision=DATASET_REVISION)
    split_names_by_config[config_name] = split_names
    print(f"Splits in {config_name!r}: {split_names}")

if CONFIG_NAME not in config_names:
    raise ValueError(f"Configuration {CONFIG_NAME!r} is unavailable. Choose one from {config_names}.")
if SPLIT_NAME not in split_names_by_config[CONFIG_NAME]:
    raise ValueError(f"Split {SPLIT_NAME!r} is unavailable for {CONFIG_NAME!r}.")

## 3. Stream a small sample

Streaming avoids downloading the entire dataset. Keep all fields so source and licence metadata remain available.

In [ ]:
dataset_stream = load_dataset(
    DATASET_ID,
    name=CONFIG_NAME,
    split=SPLIT_NAME,
    revision=DATASET_REVISION,
    streaming=True,
)

raw_records = list(islice(iter(dataset_stream), SAMPLE_LIMIT))
raw_df = pd.DataFrame.from_records(raw_records)

print(f"Records loaded: {len(raw_df):,}")
print("Columns:", list(raw_df.columns))
display(raw_df.dtypes.to_frame(name="dtype"))
display(raw_df.head())

if "text" not in raw_df.columns:
    raise KeyError(f"Expected a 'text' column, but found: {list(raw_df.columns)}")

## 4. View sample Sanskrit text

The records are displayed without changing the source sample. Metadata stays in `raw_df` for review.

In [ ]:
text_column = raw_df["text"]
for row_number, text_value in enumerate(text_column.head(10), start=1):
    print(f"{row_number}. {text_value!r}")

if "licence" in raw_df.columns:
    print("\nLicence values in this sample:")
    display(raw_df["licence"].value_counts(dropna=False).to_frame(name="records"))
if "attribution" in raw_df.columns:
    print("Attribution field is present and will be retained in the processed copy.")

## 5. Inspect Unicode characters

The frequency table shows each character, its Unicode code point, name, and count. Rare characters are review signals, not automatic errors.

In [ ]:
character_counts = Counter()
for text_value in text_column:
    if isinstance(text_value, str):
        character_counts.update(text_value)

unicode_rows = []
for character, count in character_counts.most_common(40):
    unicode_rows.append({
        "character": repr(character),
        "code_point": f"U+{ord(character):04X}",
        "unicode_name": unicodedata.name(character, "<unnamed>"),
        "count": count,
    })

display(pd.DataFrame(unicode_rows))

## 6. Define text checks

The script check covers Devanagari, Vedic Extensions, and Devanagari Extended-A. Mixed script means Devanagari letters and letters from another script appear in the same text; this is a review signal, not an automatic error. Suspicious control and unassigned characters are reported separately.

In [ ]:
DEVANAGARI_RANGES = (
    (0x0900, 0x097F),  # Devanagari
    (0x1CD0, 0x1CFF),  # Vedic Extensions
    (0x11B00, 0x11B5F),  # Devanagari Extended-A
)
JOIN_CONTROLS = {"\u200c", "\u200d"}

def is_devanagari_character(character):
    code_point = ord(character)
    for start, end in DEVANAGARI_RANGES:
        if start <= code_point <= end:
            return True
    return False

def script_flags(text_value):
    if not isinstance(text_value, str):
        return False, False, False
    letters = [ch for ch in text_value if unicodedata.category(ch).startswith("L")]
    has_devanagari = any(is_devanagari_character(ch) for ch in letters)
    has_other_letters = any(not is_devanagari_character(ch) for ch in letters)
    return has_devanagari, has_other_letters, has_devanagari and has_other_letters

def suspicious_characters(text_value):
    if not isinstance(text_value, str):
        return []
    found = set()
    for ch in text_value:
        category = unicodedata.category(ch)
        if ch in {"\n", "\r", "\t"} or ch in JOIN_CONTROLS:
            continue
        if category in {"Cc", "Cs", "Co", "Cn", "Cf"}:
            found.add(ch)
    return sorted(found, key=ord)

def is_empty_text(text_value):
    return isinstance(text_value, str) and text_value.strip() == ""

unusual_rows = []
for character, count in character_counts.items():
    category = unicodedata.category(character)
    in_devanagari_range = any(
        start <= ord(character) <= end for start, end in DEVANAGARI_RANGES
    )
    is_non_devanagari_letter = category.startswith("L") and not in_devanagari_range
    is_suspicious_category = category in {"Cc", "Cs", "Co", "Cn", "Cf"}
    is_rare_nonspace = count <= 5 and not character.isspace()

    if is_non_devanagari_letter or is_suspicious_category or is_rare_nonspace:
        unusual_rows.append({
            "character": repr(character),
            "code_point": f"U+{ord(character):04X}",
            "unicode_name": unicodedata.name(character, "<unnamed>"),
            "category": category,
            "count": count,
        })

unusual_characters_df = pd.DataFrame(unusual_rows)
if not unusual_characters_df.empty:
    unusual_characters_df = unusual_characters_df.sort_values(
        ["count", "code_point"], ascending=[True, True]
    )
print("Uncommon, non-Devanagari-letter, or control characters for review:")
display(unusual_characters_df.head(100))


## 7. Check Unicode, scripts, punctuation, whitespace, and suspicious characters

These checks annotate records for review. They do not remove or correct anything in the raw sample.

In [ ]:
text_values = raw_df["text"].tolist()
text_series = pd.Series(text_values, dtype="object")
duplicate_mask = text_series.duplicated(keep="first") & text_series.notna()

record_checks = []
for row_number, text_value in enumerate(text_values):
    is_string = isinstance(text_value, str)
    if is_string:
        has_devanagari, has_other_letters, is_mixed = script_flags(text_value)
        unusual_controls = suspicious_characters(text_value)
        row_checks = {
            "row_number": row_number,
            "is_non_string": False,
            "is_empty_or_whitespace_only": is_empty_text(text_value),
            "is_nfc": unicodedata.is_normalized("NFC", text_value),
            "has_devanagari_letters": has_devanagari,
            "has_non_devanagari_letters": has_other_letters,
            "has_mixed_scripts": is_mixed,
            "has_danda": "।" in text_value,
            "has_double_danda": "॥" in text_value,
            "has_leading_or_trailing_whitespace": text_value != text_value.strip(),
            "has_repeated_ascii_spaces": re.search(r" {2,}", text_value) is not None,
            "has_tab": "\t" in text_value,
            "has_line_break": "\n" in text_value or "\r" in text_value,
            "has_nonstandard_whitespace": any(
                ch.isspace() and ch not in " \t\r\n" for ch in text_value
            ),
            "suspicious_characters": [f"U+{ord(ch):04X}" for ch in unusual_controls],
        }
    else:
        row_checks = {
            "row_number": row_number,
            "is_non_string": True,
            "is_empty_or_whitespace_only": False,
            "is_nfc": False,
            "has_devanagari_letters": False,
            "has_non_devanagari_letters": False,
            "has_mixed_scripts": False,
            "has_danda": False,
            "has_double_danda": False,
            "has_leading_or_trailing_whitespace": False,
            "has_repeated_ascii_spaces": False,
            "has_tab": False,
            "has_line_break": False,
            "has_nonstandard_whitespace": False,
            "suspicious_characters": [],
        }

    row_checks["is_duplicate_text_after_first"] = bool(duplicate_mask.iloc[row_number])
    record_checks.append(row_checks)

checks_df = pd.DataFrame(record_checks)
print(f"Rows checked: {len(checks_df):,}")
display(checks_df.head(10))

## 8. Exact duplicate records

A duplicate is an exact repeated text value after its first occurrence. Punctuation or whitespace variants are not treated as duplicates.

In [ ]:
duplicate_records = raw_df.loc[duplicate_mask].copy()
print(f"Exact duplicate text rows after first occurrence: {len(duplicate_records):,}")
if not duplicate_records.empty:
    display(duplicate_records.head(10))

## 9. Sample quality report

These counts describe only the loaded sample. They are not estimates of the full corpus unless the sample is representative.

In [ ]:
quality_report = [
    ("records_loaded", len(raw_df)),
    ("columns_available", len(raw_df.columns)),
    ("records_with_missing_text", int(text_series.isna().sum())),
    ("records_with_non_string_text", int(checks_df["is_non_string"].sum())),
    ("records_with_empty_or_whitespace_only_text", int(checks_df["is_empty_or_whitespace_only"].sum())),
    ("unique_non_missing_texts", int(text_series.dropna().nunique())),
    ("exact_duplicate_text_rows_after_first", int(checks_df["is_duplicate_text_after_first"].sum())),
    ("string_records_not_in_nfc", sum(isinstance(value, str) and not unicodedata.is_normalized("NFC", value) for value in text_values)),
    ("records_with_devanagari_letters", int(checks_df["has_devanagari_letters"].sum())),
    ("records_with_mixed_scripts", int(checks_df["has_mixed_scripts"].sum())),
    ("records_with_danda", int(checks_df["has_danda"].sum())),
    ("records_with_double_danda", int(checks_df["has_double_danda"].sum())),
    ("records_with_leading_or_trailing_whitespace", int(checks_df["has_leading_or_trailing_whitespace"].sum())),
    ("records_with_repeated_ascii_spaces", int(checks_df["has_repeated_ascii_spaces"].sum())),
    ("records_with_tabs", int(checks_df["has_tab"].sum())),
    ("records_with_line_breaks", int(checks_df["has_line_break"].sum())),
    ("records_with_nonstandard_whitespace", int(checks_df["has_nonstandard_whitespace"].sum())),
    ("records_with_suspicious_characters", int(checks_df["suspicious_characters"].map(bool).sum())),
]
quality_report_df = pd.DataFrame(quality_report, columns=["measure", "count"])
display(quality_report_df)

## 10. Save a separate normalized sample

The shared function from `src/normalization/text_normalizer.py` is used here. The raw records remain unchanged in `raw_df`. A copy renames the original text field to `raw_text`, adds `normalized_text`, and retains all other fields. This pass applies NFC, collapses horizontal whitespace, standardizes CR/LF line endings to LF, and preserves line breaks and punctuation. The Parquet output goes under `datasets/processed/`, separate from the source corpus.

In [ ]:
processed_df = raw_df.copy().rename(columns={"text": "raw_text"})
processed_df["normalized_text"] = processed_df["raw_text"].map(normalize_text)

changed_text_count = int(
    (processed_df["raw_text"] != processed_df["normalized_text"]).fillna(False).sum()
)
print(f"Sample records: {len(processed_df):,}")
print(f"Text values changed by normalization: {changed_text_count:,}")

processed_directory = PROJECT_ROOT / "datasets" / "processed"
processed_directory.mkdir(parents=True, exist_ok=True)
output_path = processed_directory / f"sansar_{SPLIT_NAME}_sample_{len(processed_df)}_normalized.parquet"
processed_df.to_parquet(output_path, index=False)
print(f"Saved separate processed sample to: {output_path}")
display(processed_df[["raw_text", "normalized_text"]].head())

## Interpretation and limitations

- This sample comes from the beginning of a streaming split and may not represent the full corpus. Record a documented sampling strategy for later experiments.
- A non-Devanagari letter, mixed-script row, or uncommon Unicode character is a review flag, not proof of corruption.
- Normalization preserves raw text and metadata. It does not attempt OCR correction, rewrite punctuation, or infer Sanskrit word boundaries.
- The corpus page reports per-record licence and attribution metadata. Preserve these fields and review applicable terms before publication.